# 03 — Model Comparison and Results

Loads the artifacts a pipeline run produced and interrogates them. Run
`python scripts/run_pipeline.py --config configs/default.yaml` first.

In [ ]:
%load_ext autoreload
%autoreload 2

import json, pandas as pd, numpy as np
from battery_rul.config import load_config
from battery_rul.utils.io import load_json, read_table
from battery_rul.visualization import apply_style

cfg = load_config('../configs/default.yaml'); apply_style(cfg.viz)
metrics = load_json(cfg.paths.reports_dir / 'metrics.json')
print('champion:', metrics['champion'], '| selected on', metrics['selected_on'])

## The comparison table

In [ ]:
pd.DataFrame(metrics['comparison'])

### Like-for-like

Sequence models cannot score a cell's first *w*−1 cycles, and those early rows
are the hardest. This table restricts every model to rows all of them can
score, so the ranking reflects the models rather than their input requirements.

In [ ]:
pd.DataFrame(metrics.get('comparison_common_rows', []))

## Per-cell breakdown

With two test cells, this table carries more information than any aggregate.
A model that looks fine overall can be badly wrong on one cell.

In [ ]:
champ = metrics['champion']
pd.DataFrame(metrics['test'][champ]['per_battery'])

## Predicted vs true

In [ ]:
preds = read_table(cfg.paths.reports_dir / 'predictions_test.parquet')
from battery_rul.visualization.style import figure, battery_palette

d = preds[preds.model == champ].dropna(subset=['y_pred'])
colours = battery_palette(sorted(d.battery_id.unique()))
with figure(nrows=1, ncols=2, figsize=(13, 5.5), cfg=cfg.viz) as (fig, axes):
    ax0, ax1 = axes
    lim = float(max(d.y_true.max(), d.y_pred.max())) * 1.05
    ax0.plot([0, lim], [0, lim], ls='--', color='#444')
    for c, g in d.groupby('battery_id'):
        ax0.scatter(g.y_true, g.y_pred, s=20, alpha=.8, color=colours[c], label=c)
        ax1.plot(g.cycle_index, g.y_true, color='#333', lw=2)
        ax1.plot(g.cycle_index, g.y_pred, color=colours[c], ls='--', label=c)
    ax0.set_xlabel('True RUL'); ax0.set_ylabel('Predicted RUL'); ax0.legend()
    ax0.set_title(f'{champ}: predicted vs true')
    ax1.set_xlabel('Cycle'); ax1.set_ylabel('RUL'); ax1.legend()
    ax1.set_title('Trajectories (solid = truth)')

## Where the error lives

Error grows with remaining life: a fresh cell looks nearly identical whether it
will last 120 or 160 cycles. This is the central difficulty of the problem, not
a defect of the model.

In [ ]:
expl = load_json(cfg.paths.reports_dir / 'explainability.json')
pd.DataFrame(expl['error_analysis']['by_rul_band'])

## Importance by physical signal family

The collinearity-robust reading. Individual feature rankings should not be
over-read: a 5-cycle and a 10-cycle rolling mean of the same signal are
near-redundant, and attribution methods split credit between them arbitrarily.

In [ ]:
pd.Series(expl['family_importance']).sort_values(ascending=False)

## Serving reproduces training

The persisted pipeline + model, applied to raw cycles, must reproduce the
numbers the evaluator reported. If this drifts, the model is broken in
production while looking fine in the report.

In [ ]:
from battery_rul.pipelines.predict import RULPredictor
from battery_rul.pipelines.prepare_data import load_prepared

prepared = load_prepared(cfg)
cycles = prepared.cycles[prepared.cycles.battery_id.isin(prepared.split.test_batteries)]
served = RULPredictor.from_artifacts(cfg).predict(cycles)
served.per_battery

In [ ]:
merged = (d.merge(served.predictions, on=['battery_id','cycle_index'])
           .dropna(subset=['y_pred','predicted_rul_cycles']))
delta = (merged.y_pred - merged.predicted_rul_cycles).abs().max()
print(f'max |training - serving| over {len(merged)} rows: {delta:.2e} cycles')

---
See `reports/evaluation_report.md` for the full written report, and
`docs/limitations.md` before quoting any of these numbers.